In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver # store checkpoints in memory in RAM.

In [2]:
import os 
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
llm=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000204C2063A70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000204C2134A70>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.',
 'explanation': 'This joke is a play on words, which is a common technique used in puns. A pun is a form of wordplay that exploits multiple meanings of a word or phrase.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. \n\nOn one hand, "crusty" can describe the texture of a pizza crust, which is a common characteristic of a pizza.\n\nOn the other hand, "feeling a little crusty" is also an idiom that means to feel a bit grumpy or irritable, like having a bad mood.\n\nThe joke is funny because it takes the phrase "feeling a little crusty" and applies it to a pizza, using the literal meaning of "crusty" to describe the pizza\'s texture, while also implying that the pizza is feeling grumpy, which is the common usage of the phrase.\n\nThe humor comes from the unexpected twist on the phrase and the clever use of wordplay to create a pun.'}

In [ ]:
workflow.get_state(config1)  #last state of the workflow execution with thread_id "1"

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, which is a common technique used in puns. A pun is a form of wordplay that exploits multiple meanings of a word or phrase.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. \n\nOn one hand, "crusty" can describe the texture of a pizza crust, which is a common characteristic of a pizza.\n\nOn the other hand, "feeling a little crusty" is also an idiom that means to feel a bit grumpy or irritable, like having a bad mood.\n\nThe joke is funny because it takes the phrase "feeling a little crusty" and applies it to a pizza, using the literal meaning of "crusty" to describe the pizza\'s texture, while also implying that the pizza is feeling grumpy, which is the common usage of the phrase.\n\nThe humor comes from the unexpected twist on the phrase and the clever use of wordplay to create a pun.'},

In [ ]:
list(workflow.get_state_history(config1)) # list of all states during the workflow execution with thread_id "1"

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, which is a common technique used in puns. A pun is a form of wordplay that exploits multiple meanings of a word or phrase.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. \n\nOn one hand, "crusty" can describe the texture of a pizza crust, which is a common characteristic of a pizza.\n\nOn the other hand, "feeling a little crusty" is also an idiom that means to feel a bit grumpy or irritable, like having a bad mood.\n\nThe joke is funny because it takes the phrase "feeling a little crusty" and applies it to a pizza, using the literal meaning of "crusty" to describe the pizza\'s texture, while also implying that the pizza is feeling grumpy, which is the common usage of the phrase.\n\nThe humor comes from the unexpected twist on the phrase and the clever use of wordplay to create a pun.'}

In [ ]:
config2 = {"configurable": {"thread_id": "2"}} #change thread_id to "2" to start a new workflow execution from the start node, without affecting the previous workflow execution with thread_id "1"
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to therapy? \n\nBecause it was feeling a little "drained".',
 'explanation': 'The joke "Why did the pasta go to therapy? Because it was feeling a little \'drained\'" is a play on words. It uses a pun to connect the phrase "feeling drained" to the fact that pasta is often served with a sauce that can leave it feeling empty or depleted of its liquid content.\n\nIn a literal sense, when we say someone is "feeling drained," it means they are physically or emotionally exhausted. However, in the context of the joke, the phrase is used to make a pun on the fact that pasta can become dry or empty when it absorbs a sauce. The punchline is humorous because it takes the common phrase "feeling drained" and gives it a new, unexpected meaning related to the characteristics of pasta.\n\nThis type of wordplay is often used in humor to create a unexpected connection between two ideas, and the joke relies on the listener being familiar with both the comm

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, which is a common technique used in puns. A pun is a form of wordplay that exploits multiple meanings of a word or phrase.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. \n\nOn one hand, "crusty" can describe the texture of a pizza crust, which is a common characteristic of a pizza.\n\nOn the other hand, "feeling a little crusty" is also an idiom that means to feel a bit grumpy or irritable, like having a bad mood.\n\nThe joke is funny because it takes the phrase "feeling a little crusty" and applies it to a pizza, using the literal meaning of "crusty" to describe the pizza\'s texture, while also implying that the pizza is feeling grumpy, which is the common usage of the phrase.\n\nThe humor comes from the unexpected twist on the phrase and the clever use of wordplay to create a pun.'},

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, which is a common technique used in puns. A pun is a form of wordplay that exploits multiple meanings of a word or phrase.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. \n\nOn one hand, "crusty" can describe the texture of a pizza crust, which is a common characteristic of a pizza.\n\nOn the other hand, "feeling a little crusty" is also an idiom that means to feel a bit grumpy or irritable, like having a bad mood.\n\nThe joke is funny because it takes the phrase "feeling a little crusty" and applies it to a pizza, using the literal meaning of "crusty" to describe the pizza\'s texture, while also implying that the pizza is feeling grumpy, which is the common usage of the phrase.\n\nThe humor comes from the unexpected twist on the phrase and the clever use of wordplay to create a pun.'}

# Time Travel

In [14]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, which is a common technique used in puns. A pun is a form of wordplay that exploits multiple meanings of a word or phrase.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. \n\nOn one hand, "crusty" can describe the texture of a pizza crust, which is a common characteristic of a pizza.\n\nOn the other hand, "feeling a little crusty" is also an idiom that means to feel a bit grumpy or irritable, like having a bad mood.\n\nThe joke is funny because it takes the phrase "feeling a little crusty" and applies it to a pizza, using the literal meaning of "crusty" to describe the pizza\'s texture, while also implying that the pizza is feeling grumpy, which is the common usage of the phrase.\n\nThe humor comes from the unexpected twist on the phrase and the clever use of wordplay to create a pun.'}

# Updating State


In [18]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f11e55a-bbc2-6a30-8000-6d5246509881'}}

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11e55a-bbc2-6a30-8000-6d5246509881'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-12T20:54:32.748497+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='f3fa6e28-123a-e839-a3b2-3b8a77c81f6a', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, which is a common technique used in puns. A pun is a form of wordplay that exploits multiple meanings of a word or phrase.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. \n\nOn

# Fault Tolerance

In [21]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [22]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [23]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [24]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))